In [1]:
import pandas as pd

# Carregando o dataset
df = pd.read_csv('fipe_2022.csv')

print("Primeiras linhas do dataset:")
display(df.head())

print("\nInformações técnicas:")
df.info()

Primeiras linhas do dataset:


,year_of_reference,month_of_reference,fipe_code,authentication,brand,model,fuel,gear,engine_size,year_model,avg_price_brl,age_years
0,2022,January,038001-6,vwmrywl5qs,Acura,NSX 3.0,Gasoline,manual,3.0,1995,43779.0,28
1,2022,January,038001-6,t9mt723qhz,Acura,NSX 3.0,Gasoline,manual,3.0,1994,42244.0,29
2,2022,January,038001-6,tr5wv4z21g,Acura,NSX 3.0,Gasoline,manual,3.0,1993,40841.0,30
3,2022,January,038001-6,s2xxsjz3mt,Acura,NSX 3.0,Gasoline,manual,3.0,1992,39028.0,31
4,2022,January,038001-6,rtm9gj7zk8,Acura,NSX 3.0,Gasoline,manual,3.0,1991,35678.0,32



Informações técnicas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290275 entries, 0 to 290274
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   year_of_reference   290275 non-null  int64  
 1   month_of_reference  290275 non-null  object 
 2   fipe_code           290275 non-null  object 
 3   authentication      290275 non-null  object 
 4   brand               290275 non-null  object 
 5   model               290275 non-null  object 
 6   fuel                290275 non-null  object 
 7   gear                290275 non-null  object 
 8   engine_size         290275 non-null  float64
 9   year_model          290275 non-null  int64  
 10  avg_price_brl       290275 non-null  float64
 11  age_years           290275 non-null  int64  
dtypes: float64(2), int64(3), object(7)
memory usage: 26.6+ MB


In [2]:
# 1. Filtrar para Janeiro para evitar repetições do mesmo carro em meses diferentes
df_janeiro = df[df['month_of_reference'] == 'January'].copy()

# 2. Selecionar features essenciais (incluindo 'model' para exibição e 'age_years' para depreciação)
features_list = ['brand', 'model', 'fuel', 'gear', 'engine_size', 'year_model', 'avg_price_brl', 'age_years']
df_filtered = df_janeiro[features_list].copy().reset_index(drop=True)

print(f"Dataset pronto para processamento: {len(df_filtered)} veículos únicos.")

Dataset pronto para processamento: 24031 veículos únicos.


In [3]:
!pip install scikit-learn

In [4]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.compose import ColumnTransformer


# 1. Definir o que é número e o que é categoria
num_features = ['engine_size', 'year_model', 'avg_price_brl', 'age_years']
cat_features = ['fuel', 'gear']

# 2. Criar o transformador (Normaliza números e converte textos em colunas binárias)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(), cat_features)
])

# 3. Gerar a matriz matemática
matrix_transformed = preprocessor.fit_transform(df_filtered)

print("Matriz preparada para o motor de recomendação!")
print(f"Formato da matriz: {matrix_transformed.shape}")

Matriz preparada para o motor de recomendação!
Formato da matriz: (24031, 9)


In [5]:
def recomendar_concorrentes(id_carro_escolhido, top_n=10, margem_preco=0.25):
    # 1. Isolar o vetor do carro que o usuário escolheu
    carro_vetor = matrix_transformed[id_carro_escolhido].reshape(1, -1)
    
    # 2. Calcular a similaridade contra todos os outros carros do banco
    sim_scores = cosine_similarity(carro_vetor, matrix_transformed).flatten()
    
    # 3. Preparar o DataFrame de análise
    df_analise = df_filtered.copy()
    df_analise['score_similaridade'] = sim_scores
    
    # Dados do carro de referência
    carro_ref = df_filtered.iloc[id_carro_escolhido]
    preco_ref = carro_ref['avg_price_brl']
    marca_ref = carro_ref['brand']
    
    # 4. Regras de Negócio para o Cliente:
    # - Não sugerir a mesma marca (queremos concorrentes)
    # - Manter o preço em uma margem aceitável (default 25%)
    filtro_marcas = df_analise['brand'] != marca_ref
    preco_min, preco_max = preco_ref * (1 - margem_preco), preco_ref * (1 + margem_preco)
    filtro_preco = df_analise['avg_price_brl'].between(preco_min, preco_max)
    
    # 5. Filtrar e ordenar pelos mais parecidos
    recomendacoes = df_analise[filtro_marcas & filtro_preco].sort_values(by='score_similaridade', ascending=False)
    
    return carro_ref, recomendacoes.head(top_n)

In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Filtro de Janeiro e inclusao da coluna 'model'
df_janeiro = df[df['month_of_reference'] == 'January'].copy()

# Lista de colunas necessarias (incluindo 'model' para a interface)
features_list = ['brand', 'model', 'fuel', 'gear', 'engine_size', 'year_model', 'avg_price_brl', 'age_years']
df_filtered = df_janeiro[features_list].copy().reset_index(drop=True)

# 2. Re-treinar a matriz para garantir o tamanho correto (24031 linhas)
num_features = ['engine_size', 'year_model', 'avg_price_brl', 'age_years']
cat_features = ['fuel', 'gear']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(), cat_features)
])

matrix_transformed = preprocessor.fit_transform(df_filtered)

print("Dados preparados com sucesso. Coluna 'model' incluida.")

Dados preparados com sucesso. Coluna 'model' incluida.


In [7]:
from IPython.display import display, HTML, clear_output

# Garante que o perfil de busca esteja pronto
df_filtered['search_profile'] = (
    df_filtered['brand'].astype(str) + " " + 
    df_filtered['model'].astype(str) + " " + 
    df_filtered['year_model'].astype(str) + " " + 
    df_filtered['engine_size'].astype(str)
).str.lower()

# Entrada do usuário
print("-" * 60)
entrada = input("🔍 O que você procura? (Ex: 'honda 2020 1.0'): ").lower()
print("-" * 60)

termos = entrada.split()
matches = df_filtered.copy()

for termo in termos:
    matches = matches[matches['search_profile'].str.contains(termo, na=False)]

if matches.empty:
    print(f"❌ Nenhum veículo encontrado para '{entrada}'. Tente algo como 'Civic' ou 'Fiat'.")
else:
    display(HTML("<h4>📍 Selecione a versão exata:</h4>"))
    exibir = matches.head(10)
    for i, (idx, row) in enumerate(exibir.iterrows()):
        print(f" [{i}] {row['brand'].upper()} {row['model']} ({row['year_model']}) - R$ {row['avg_price_brl']:,.2f}")
    
    try:
        escolha = int(input("\n👉 Digite o número da sua escolha: "))
        id_escolhido = exibir.index[escolha]
        
        # Gerar as 10 Recomendações
        alvo, sugestoes = recomendar_concorrentes(id_escolhido, top_n=10)
        clear_output(wait=True)

        # Cabeçalho Estético
        header = f"""
        <div style="background-color: #1a2a6c; padding: 20px; border-radius: 12px; color: white; font-family: sans-serif;">
            <h2 style="margin:0; color: #fdbb2d;">SAUTER DIGITAL | Inteligência de Mercado</h2>
            <p style="font-size: 16px;">Análise para: <b>{alvo['brand']} {alvo['model']} ({alvo['year_model']})</b></p>
            <div style="display: flex; gap: 30px; opacity: 0.9; font-size: 14px;">
                <span>💰 Preço: R$ {alvo['avg_price_brl']:,.2f}</span>
                <span>⚙️ Motor: {alvo['engine_size']}</span>
            </div>
        </div>
        <br>
        """
        display(HTML(header))

        # Tabela Estruturada
        df_table = sugestoes[['brand', 'model', 'year_model', 'engine_size', 'avg_price_brl', 'score_similaridade']].copy()
        df_table.columns = ['Marca', 'Modelo', 'Ano', 'Motor', 'Preço (R$)', 'Afinidade']
        
        styled_table = df_table.style.format({
            'Preço (R$)': 'R$ {:,.2f}',
            'Afinidade': '{:.1%}'
        }).bar(subset=['Afinidade'], color='#2ecc71', vmin=0, vmax=1)\
          .set_properties(**{'text-align': 'left', 'padding': '12px', 'border': '1px solid #eee'})\
          .set_table_styles([
              {'selector': 'th', 'props': [('background-color', '#1a2a6c'), ('color', 'white'), ('font-weight', 'bold')]},
              {'selector': 'tr:hover', 'props': [('background-color', '#f9f9f9')]}
          ]).hide(axis='index')

        display(styled_table)
        
    except (ValueError, IndexError):
        print("❌ Escolha inválida. Por favor, digite um dos números da lista acima.")

Marca,Modelo,Ano,Motor,Preço (R$),Afinidade
VW - VolksWagen,Gol 1.6 Mi/ Power 1.6 Mi 4p,2002,1.600000,"R$ 12,652.00",100.0%
Ford,Fiesta GLX 1.6 8V 5p,2002,1.600000,"R$ 12,460.00",100.0%
VW - VolksWagen,Grand Saveiro Xtreme/Street 1.6/1.8/2.0,2002,1.600000,"R$ 12,909.00",100.0%
Fiat,Palio ELX 1.6 mpi 16v 4p,2002,1.600000,"R$ 12,411.00",100.0%
Ford,Fiesta GLX 1.6 8V 3p,2002,1.600000,"R$ 12,407.00",100.0%
Ford,Fiesta Street 1.6 8v 95cv 5p,2002,1.600000,"R$ 12,238.00",100.0%
Seat,Inca 1.6L,2002,1.600000,"R$ 12,104.00",100.0%
Peugeot,206 Rallye 1.6/ 1.6 Flex 16v 110cv 3p,2002,1.600000,"R$ 11,951.00",100.0%
GM - Chevrolet,Corsa Wind 1.6 MPFi 2p,2002,1.600000,"R$ 11,921.00",100.0%
Fiat,Siena ELX 1.6 mpi 8V/16V 4p,2002,1.600000,"R$ 11,864.00",100.0%
